# 학생용 워크북 1: 셀 0부터 순차 실행
노션 지시서의 같은 실습 번호를 따라 TODO를 수정하고 실행합니다.
PDF 업로드는 실습 1에서 수행합니다. 실습 5의 선택형 질문도 A/B/C로 맞춰야 합니다.


## 실습 0
노션의 동일한 실습 번호 설명·예제·TODO·예상 결과를 확인하고 바로 아래 셀을 실행하세요.


In [ ]:
%pip -q install pypdf scikit-learn


## 실습 1
노션의 동일한 실습 번호 설명·예제·TODO·예상 결과를 확인하고 바로 아래 셀을 실행하세요.


## 자료를 직접 선택하세요
링크에서 PDF 파일 1개를 내려받습니다. 다음 코드 셀의 파일 업로드 창에 해당 PDF 1개만 선택하세요.
- [A 안내서](http://www.doczoom.co.kr/dist/manual/DocZoomCreatorPro_QuickStartGuide_ko.pdf)
- [B 안전관리계획](https://www.du.ac.kr/resources/AD/main/file/2026%ED%95%99%EB%85%84%EB%8F%84%20%EC%95%88%EC%A0%84%EA%B4%80%EB%A6%AC%20%EA%B3%84%ED%9A%8D%EC%84%9C.pdf)
- [C 연구보고서](https://repo.kicce.re.kr/bitstream/2019.oak/5161/2/CR2102.pdf)
페이지 수가 많으면 청크 수와 처리 시간이 늘어납니다.


In [ ]:
from pathlib import Path
from pypdf import PdfReader

# 실습 1: 제공된 자료 A/B/C 중 하나를 직접 내려받아 Colab에 업로드합니다.
# 반드시 본인이 선택한 PDF 한 개만 현재 작업 파일로 지정합니다.
from google.colab import files
uploaded = files.upload()
pdf_files = [name for name in uploaded if name.lower().endswith(".pdf")]
assert len(pdf_files) == 1, "PDF 한 개만 업로드하세요."
pdf_name = pdf_files[0]
pdf_path = Path(pdf_name)
assert pdf_path.exists()
print("선택한 PDF:", pdf_name)

reader = PdfReader(str(pdf_path))
records = []
for page_number, page in enumerate(reader.pages, start=1):
    text = (page.extract_text() or "").strip()
    if text:
        records.append({"page": page_number, "text": text})

assert records, "추출 텍스트가 없습니다. 스캔 PDF라면 OCR이 필요합니다."
print("텍스트가 있는 페이지:", len(records))\nprint("선택 문서:", pdf_name)


## 실습 2
노션의 동일한 실습 번호 설명·예제·TODO·예상 결과를 확인하고 바로 아래 셀을 실행하세요.


In [ ]:
def chunk_words(text, size=75, overlap=20):
    if size <= 0 or overlap < 0 or overlap >= size:
        raise ValueError("0 <= overlap < size 조건을 확인하세요.")
    words = text.split()
    step = size - overlap
    return [
        " ".join(words[i:i + size])
        for i in range(0, len(words), step)
        if words[i:i + size]
    ]

chunk_size = None  # TODO 2: 75
overlap = None  # TODO 2: 20
chunks = [
    {"page": item["page"], "text": part}
    for item in records
    for part in chunk_words(item["text"], chunk_size, overlap)
]
print("생성 청크:", len(chunks))
print("첫 청크:", chunks[0]["text"][:100])


## 실습 3
노션의 동일한 실습 번호 설명·예제·TODO·예상 결과를 확인하고 바로 아래 셀을 실행하세요.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    analyzer="word"  # TODO 3: char로 변경,
    ngram_range=(1, 1)  # TODO 3: (2, 4)로 변경
)
document_vectors = vectorizer.fit_transform(
    [part["text"] for part in chunks]
)
print("행렬 크기:", document_vectors.shape)


## 실습 4
노션의 동일한 실습 번호 설명·예제·TODO·예상 결과를 확인하고 바로 아래 셀을 실행하세요.


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(question, top_k=3):
    question_vector = vectorizer.transform([question])
    scores = cosine_similarity(  # TODO 4: 유사도 계산 함수를 설명하세요

        question_vector, document_vectors
    ).ravel()
    order = np.argsort(scores)[::-1][:top_k]
    return [
        {
            "page": chunks[int(i)]["page"],
            "text": chunks[int(i)]["text"],
            "score": float(scores[int(i)])
        }
        for i in order
    ]

results = retrieve("RAG가 무엇인가요?", top_k=2)
for item in results:
    print(f"페이지 {item['page']} / 점수 {item['score']:.3f}")
    print(item["text"][:100])


## 실습 5
노션의 동일한 실습 번호 설명·예제·TODO·예상 결과를 확인하고 바로 아래 셀을 실행하세요.


In [ ]:
import pandas as pd

# 선택 PDF: 자료 1/4 = A, 자료 2 = B, 자료 3 = C
selected_type = "B"  # TODO 5: 본인이 선택한 자료에 맞춰 A/B/C로 변경
question_sets = {
    "A": [
        "설치와 시작 절차는 무엇인가요?",
        "프로그램의 주요 기능은 무엇인가요?",
        "사용 시 주의사항은 무엇인가요?"
    ],
    "B": [
        "안전관리 계획의 목적은 무엇인가요?",
        "안전관리 조직과 역할은 무엇인가요?",
        "사고 발생 시 대응 절차는 무엇인가요?"
    ],
    "C": [
        "연구의 목적은 무엇인가요?",
        "조사의 주요 내용은 무엇인가요?",
        "종합 지원체계의 개선 방향은 무엇인가요?"
    ]
}
questions = question_sets[selected_type]
rows = []
for question in questions:
    hit = retrieve(question, top_k=1)[0]
    rows.append({
        "질문": question,
        "페이지": hit["page"],
        "유사도": round(hit["score"], 3)
    })
result_table = pd.DataFrame(rows)
display(result_table)


## 결과 제출
모든 셀을 실행한 후 결과표와 실행화면을 캡처하고 TODO를 완성한 노트북을 제출하세요.
